# Function Calling 功能概述

（一）功能原理
Function Calling 允许模型根据用户的输入，判断是否需要调用外部函数来完成任务。模型通过对用户问题的理解，识别出需要借助特定函数解决的部分，然后生成调用函数的指令。<span style="color:red">这些函数可以是自定义的工具函数，也可以是调用第三方 API 的函数。</span>例如，当用户询问 “明天北京的天气如何”，模型识别到这是一个获取天气信息的任务，就可以调用相应的天气查询函数来获取数据并回答用户。


（二）与传统对话的区别
与传统的对话模型不同，Function Calling 打破了模型只能基于自身知识进行回答的限制。传统对话模型主要依赖预训练的知识来生成回复，而 Function Calling 使得模型能够动态地调用外部工具，获取最新的信息和执行特定的任务，从而提供更准确、更实用的回答。


## 二、Function Calling使用方法详解

（一）定义业务函数
在使用 Function Calling 之前，开发者需要定义可供模型调用的函数。这些函数需要有清晰的定义，包括函数名、参数列表和功能描述。例如，定义一个获取城市天气的函数：

In [9]:
def get_weather(city):
    """
    获取指定城市的天气信息
    :param city: 城市名称
    :return: 包含天气状况和温度的字典
    """
    # 这里是实际获取天气数据的代码，例如调用天气API
    weather_data = {
        "condition": "sunny",
        "temperature": "25°C"
    }
    return weather_data


## （二）初始化大模型客户端
导入相关的依赖包，并初始化大模型客户端



In [10]:
import json
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# print(os.getenv("api_key"))
# print(os.getenv("base_url"))


True

In [11]:
client = OpenAI(api_key=os.getenv("api_key"), 
                base_url=os.getenv("base_url"))

## （三）定义tools工具函数


In [12]:
# 定义tools工具函数
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "获取指定城市的天气信息",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "城市名称"
                    }
                },
                "required": ["city"]
            }
        }
    }
]


## （四）第一次调用大模型API
在调用 DeepSeek API 时，需要在请求体中包含tools参数，用于指定模型可以调用的函数列表。同时，tool_choice参数可以控制模型调用工具的行为，取值包括none（不调用任何工具，直接生成消息）、auto（模型可选择生成消息或调用工具）、required（模型必须调用工具）。


In [19]:
# 定义消息
messages = [
    {"role": "user", "content": "明天北京的天气如何"}
]

# 第1次大模型调用（查看是否有匹配到工具函数）
response = client.chat.completions.create(
    model="deepseek-chat",
    messages=messages,
    tools=tools,
    tool_choice="auto"
)
# 查看第一次调用后返回的消息（检查大模型是否有找到函数信息）
reply = response.choices[0].message
messages.append(reply)
print(reply)


ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_3ams3jjbdbe83muidp21biyf', function=Function(arguments='{"city":"北京"}', name='get_weather'), type='function')])


## (五)调用真正的业务函数


In [14]:
reply.tool_calls[0].function.arguments

'{"city":"北京"}'

In [15]:
reply.tool_calls[0].function.name

'get_weather'

In [16]:
function_args = json.loads(reply.tool_calls[0].function.arguments)  # 确保转换为字典
result = get_weather(**function_args) #真正调用，手动执行获取天气信息的方法
print(result)


{'condition': 'sunny', 'temperature': '25°C'}


## （六）第二次调用大模型API
第2次大模型调用；将获取到的结果，丢给大模型帮忙整理后再输出给用户。
示例代码如下：

In [17]:
# messages.append({"role": "tool", "name": function_name, "content": str(result)})
tool_call_id = reply.tool_calls[0].id
#print("tool_call_id",tool_call_id)
messages.append({"role": "tool", "tool_call_id": tool_call_id, "content": str(result)})
second_response = client.chat.completions.create(
    model="deepseek-chat",
    messages=messages
)
print(second_response.choices[0].message.content)


明天北京的天气预计为晴天，气温在25°C左右，适合户外活动。记得做好防晒和补水哦！


## 四、Function Calling应用场景

### （一）智能助手
在智能助手应用中，Function Calling 可以使助手调用各种工具函数，如查询日历、发送邮件、设置提醒等。例如，用户说 “帮我明天下午三点设置一个会议提醒”，智能助手可以调用设置提醒的函数来完成任务。

### （二）数据分析
在数据分析场景下，模型可以调用数据分析函数，如数据清洗、统计分析、数据可视化等。当用户询问 “对这个销售数据进行一下统计分析”，模型可以调用相应的数据分析函数，并将分析结果以可视化的方式呈现给用户。

### （三）电商购物
在电商购物平台中，Function Calling 可以帮助用户查询商品信息、下单购买、跟踪物流等。用户说 “我想购买一部苹果手机”，模型可以调用商品查询和下单函数，完成购物流程。


## 五、Function Calling开发注意事项
### （一）函数定义规范
定义函数时，要确保函数的描述清晰准确，参数定义明确。不规范的函数定义可能导致模型错误地调用函数，或者无法正确解析函数的返回结果。

### （二）安全问题
在调用外部函数和 API 时，要注意安全问题，防止数据泄露和恶意攻击。例如，对用户输入进行严格的验证和过滤，避免 SQL 注入和其他安全漏洞。

### （三）性能优化
过多的函数调用可能会影响性能，特别是在调用第三方 API 时，可能会面临网络延迟等问题。开发者需要合理设计函数调用策略，优化性能，例如缓存常用的函数调用结果。
